# Graficas con resultados

## Configuración

In [ ]:
!pip install -q bayesian-optimization
!pip install -U hyperopt
!pip install statsforecast
!pip install neuralforecast

In [ ]:
#############
# Librerias #
#############

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import plotly.express as px
import statsmodels.api as sm
import datetime
import math
from neuralforecast import NeuralForecast
from statsforecast import StatsForecast
from statsforecast.models import AutoETS, AutoARIMA

In [ ]:
###################
# Configuraciones #
###################

# Parametros para las series de tiempo
ts_freq = 'W'
ts_freq2 = 52

# Semanas epidemiologicas de las olas
ola_1 = '2020_08_2020_39'
ola_2 = '2020_40_2021_15'
ola_3 = '2021_23_2021_42'
ola_4 = '2021_51_2022_09'
ola_5 = '2022_22_2022_33'
ola_6 = '2022_49_2023_04'

# Periodo de tiempo para pronosticar el consumo al acabar la pandemia
ola_all = '2020_12_2023_18'
ola_all2 = '2022_09_2023_18'

# Periodos para los analisis retrospectivos
ola_retro_1 = '2020_08_2020_39'
ola_retro_2 = '2020_12_2021_35'
ola_retro_3 = '2020_12_2022_09'

# Lista de variables exogenas
lista_exog = ['F_INER', 'M_INER', 'F_CASOS', 'M_CASOS', 'FLAG_OLA', 'FLAG_VAC']
lista_cons = ['FLAG_VAC']

# Medicamento a probar
ruta_med = 'MIDAZOLAM'

# Semila para replicas
np.random.seed(1)

# Listado de medicamentos
lista_med = ["FENTANIL", "MIDAZOLAM", "PARACETAMOL"]

# Quantil contemplado para los pronosticos (en caso de ser requerido)
aux_ql = 0.75

# Listado de funciones de perdida
#lista_loss = [MAPE(), MASE(1), QuantileLoss(aux_ql)]

# Listado de periodos de tiempo a pronosticar
lista_per = [1, 5, 12]

# Listado de modelos
lista_mod = ["AutoETS", "AutoARIMA", "LSTM", "DeepAR", "TFT", "Informer", "Autoformer", "TCN"]

In [ ]:
# Carga de datos de pacientes
# datos_econo = pd.read_csv()

# Carga de datos de casos confirmados
# datos_casos = pd.read_csv()

# Carga de series de tiempo de insumos medicos
# datos_cost = pd.read_csv()

# Se agrega el numero de ingresos al INER
datos_series = pd.merge(datos_cost, datos_econo, left_on = ['FECHA'], right_on = ['FECHA_INGRESO_INER'], how = 'left')
datos_series.drop(columns = ['FECHA_INGRESO_INER'], inplace = True)

# Se agrega el numero de casos
datos_series = pd.merge(datos_series, datos_casos, left_on = ['FECHA'], right_on = ['FECHA_INGRESO'], how = 'left', suffixes=('_INER', '_CASOS'))
datos_series.drop(columns = ['FECHA_INGRESO'], inplace = True)

# Se asigna la semana epidemiologica
datos_series['SEMANA_EPIDEM']  = datos_series['FECHA'].dt.isocalendar().week
datos_series.head(10)

# Se agrega una bandera para indicar la presencia de las olas de contagio
olas = [ola_1, ola_2, ola_3, ola_4, ola_5, ola_6]
for r in range(0,6):

  aux_fin = datetime.datetime.strptime(olas[r][8:12] + ' ' +  olas[r][13:17] + ' 1', "%Y %W %w")
  aux_ini = datetime.datetime.strptime(olas[r][0:4] + ' ' +  olas[r][5:7] + ' 1', "%Y %W %w")
  datos_series.loc[(aux_ini <= datos_series['FECHA']) & (datos_series['FECHA'] <= aux_fin), 'FLAG_OLA'] = 1

# Se agrega una bandera para indicar la aplicacion de la vacuna contra COVID-19:
datos_series.loc['2020/12/23' <= datos_series['FECHA'], 'FLAG_VAC'] = 1

# Se rellanan valores con Nan con 0's
datos_series.fillna(0, inplace = True)

datos_series.reset_index(drop=True)

## Series de tiempo

### Ingresos en el INER, casos registrados a nivel nacional y consumo de medicamentos

In [ ]:
# Ola a analizar
aux_ola = ola_all

aux_ola_fin = datetime.datetime.strptime(aux_ola[8:12] + ' ' +  aux_ola[13:17] + ' 1', "%Y %W %w")
aux_ola_ini = datetime.datetime.strptime(aux_ola[0:4] + ' ' +  aux_ola[5:7] + ' 1', "%Y %W %w")

# Datos
aux = datos_series.loc[(aux_ola_ini <= datos_series['FECHA']) & (datos_series['FECHA'] <= aux_ola_fin)].copy()

# Grafica
f, ax = plt.subplots(5, figsize=(15, 13))
sns.lineplot(aux, x = 'FECHA', y = 'CASOS', ax = ax[0]).set(xlabel = ' ', ylabel = 'Casos reportados', title = 'Pandemia de COVID-19 en México')
sns.lineplot(aux, x = 'FECHA', y = 'OCUPACION', ax = ax[1]).set(ylabel = 'Camas ocupadas')
sns.lineplot(aux, x = 'FECHA', y = 'FENTANIL', ax = ax[2]).set(xlabel = '')
sns.lineplot(aux, x = 'FECHA', y = 'MIDAZOLAM', ax = ax[3]).set(xlabel = '')
sns.lineplot(aux, x = 'FECHA', y = 'PARACETAMOL', ax = ax[4]).set(xlabel = 'Semana epidemiológica')


for r in olas:
  s = '#aecdc7'
  aux_ola = r
  aux_ola_fin = datetime.datetime.strptime(aux_ola[8:12] + ' ' +  aux_ola[13:17] + ' 1', "%Y %W %w")
  aux_ola_ini = datetime.datetime.strptime(aux_ola[0:4] + ' ' +  aux_ola[5:7] + ' 1', "%Y %W %w")
  if r == '2020_08_2020_39':
    ax[0].axvspan(aux['FECHA'].iloc[aux.loc[(aux_ola_ini <= aux['FECHA']) & (aux['FECHA'] <= aux_ola_fin)].index.min()], aux['FECHA'].iloc[aux.loc[(aux_ola_ini <= aux['FECHA']) & (aux['FECHA'] <= aux_ola_fin)].index.max()], alpha=0.5, color=s, label = 'Ola de contagio de COVID-19')

  ax[0].axvspan(aux['FECHA'].iloc[aux.loc[(aux_ola_ini <= aux['FECHA']) & (aux['FECHA'] <= aux_ola_fin)].index.min()], aux['FECHA'].iloc[aux.loc[(aux_ola_ini <= aux['FECHA']) & (aux['FECHA'] <= aux_ola_fin)].index.max()], alpha=0.5, color=s)
  ax[1].axvspan(aux['FECHA'].iloc[aux.loc[(aux_ola_ini <= aux['FECHA']) & (aux['FECHA'] <= aux_ola_fin)].index.min()], aux['FECHA'].iloc[aux.loc[(aux_ola_ini <= aux['FECHA']) & (aux['FECHA'] <= aux_ola_fin)].index.max()], alpha=0.5, color=s)
  ax[2].axvspan(aux['FECHA'].iloc[aux.loc[(aux_ola_ini <= aux['FECHA']) & (aux['FECHA'] <= aux_ola_fin)].index.min()], aux['FECHA'].iloc[aux.loc[(aux_ola_ini <= aux['FECHA']) & (aux['FECHA'] <= aux_ola_fin)].index.max()], alpha=0.5, color=s)
  ax[3].axvspan(aux['FECHA'].iloc[aux.loc[(aux_ola_ini <= aux['FECHA']) & (aux['FECHA'] <= aux_ola_fin)].index.min()], aux['FECHA'].iloc[aux.loc[(aux_ola_ini <= aux['FECHA']) & (aux['FECHA'] <= aux_ola_fin)].index.max()], alpha=0.5, color=s)
  ax[4].axvspan(aux['FECHA'].iloc[aux.loc[(aux_ola_ini <= aux['FECHA']) & (aux['FECHA'] <= aux_ola_fin)].index.min()], aux['FECHA'].iloc[aux.loc[(aux_ola_ini <= aux['FECHA']) & (aux['FECHA'] <= aux_ola_fin)].index.max()], alpha=0.5, color=s)
f.legend()
plt.show()

### Escenarios retrospectivos

In [ ]:
# Ola a analizar
aux_ola = ola_retro_1

aux_ola_fin = datetime.datetime.strptime(aux_ola[8:12] + ' ' +  aux_ola[13:17] + ' 1', "%Y %W %w")
aux_ola_ini = datetime.datetime.strptime(aux_ola[0:4] + ' ' +  aux_ola[5:7] + ' 1', "%Y %W %w")

# Particion de datos: entrenamiento y prueba
aux = datos_series.loc[(aux_ola_ini <= datos_series['FECHA']) & (datos_series['FECHA'] <= aux_ola_fin)].copy()
aux_fecha = aux['FECHA'].copy().reset_index(drop=True)

# Todas las variables para el entrenamiento
all_var = [ruta_med] + lista_exog

# Todas las variables para la evaluacion
all_var2 = ["PARACETAMOL", "MIDAZOLAM", "FENTANIL"] + lista_exog

# Datos para la evaluacion de los modelos
datos_eval = pd.DataFrame(aux[all_var2]).reset_index(drop=True)
datos_eval['ds'] = aux_fecha
datos_eval['unique_id'] = '1'

# Visualizacion al ajuste hecho a los datos de entrenamiento
f, ax = plt.subplots(6, figsize=(15, 15))
s = 0

#Iteracion sobre los medicamentos modelados
for w in lista_med:

  for l in ["AutoARIMA", "AutoETS"]:

    aux = datos_eval.copy()

    # Caracteristicas del escenario retrospectivo
    re_entrenar = True
    t_val = 3
    t_test = 3
    t_step = 1
    t = t_step

    if l == "AutoARIMA":
      aux_loss = 'AIC'
      sf = StatsForecast(models=[AutoARIMA(seasonal=True, ic='aicc', stationary=False)], freq='W-Mon')

      # Se obtienen los pronosticos durante el periodo de prueba
      Y_hat_insample = sf.cross_validation(h = t, df = datos_eval[['ds', w, 'unique_id']][:-t_test-t_val], step_size= t_step, test_size= t_test, refit= re_entrenar, fitted = True, target_col=w  )
      for r in range(t_val+t_test+1):
        if r == 0:
          aux_for = sf.forecast(h = t, df = datos_eval[['ds', w, 'unique_id']][:-t_test-t_val+r-1], target_col=w)
        else:
          aux_for = pd.concat([aux_for, sf.forecast(h = t, df = datos_eval[['ds', w, 'unique_id']][:-t_test-t_val-1+r], target_col=w)], ignore_index=True)
      ventanas = aux_for['ds']
      aux_real = aux[['ds', w, 'unique_id']]
      aux_for[l] = aux_for[l].apply(lambda x: math.exp(x) if x > 0 else 0)
      aux_real = pd.merge(aux_real, aux_for, on='ds', how='left')
      aux0 = aux['y'].values[:-t_test]

      # Intervalo de prueba
      aux1 = aux['y'].values[-t_test:]
      aux2 = aux_real[l].values[-t_test:]

    # Modelos de NeuralForecast
    elif l == "AutoETS":
      aux_loss = 'AIC'
      sf = StatsForecast(models=[AutoETS(model="ANN", )], freq='W-Mon')

      # Se obtienen los pronosticos durante el periodo de prueba
      Y_hat_insample = sf.cross_validation(h = t, df = datos_eval[['ds', w, 'unique_id']][:-t_test-t_val], step_size= t_step, test_size= t_test, refit= re_entrenar, fitted = True, target_col=w  )
      for r in range(t_val+t_test+1):
        if r == 0:
          aux_for = sf.forecast(h = t, df = datos_eval[['ds', w, 'unique_id']][:-t_test-t_val+r-1], target_col=w)
        else:
          aux_for = pd.concat([aux_for, sf.forecast(h = t, df = datos_eval[['ds', w, 'unique_id']][:-t_test-t_val-1+r], target_col=w)], ignore_index=True)
      ventanas = aux_for['ds']
      aux_real = aux[['ds', w, 'unique_id']]
      aux_for[l] = aux_for[l].apply(lambda x: math.exp(x) if x > 0 else 0)
      aux_real = pd.merge(aux_real, aux_for, on='ds', how='left')
      aux0 = aux['y'].values[:-t_test]

      # Intervalo de prueba
      aux1 = aux['y'].values[-t_test:]
      aux2 = aux_real[l].values[-t_test:]


    # Se grafican los datos reales y los datos pronosticados
    if (l == 'AutoETS' and w == 'FENTANIL') or (l == 'AutoETS' and w == 'MIDAZOLAM') or (l == 'AutoETS' and w == 'PARACETAMOL'):

      ax[s].plot(aux['ds'], aux['y'])
      ax[s].plot(aux_real['ds'], aux_real[l], ls='--')

      # Se sombrean los intervalos de prueba y validacion
      ax[s].axvspan(Y_hat_insample['ds'].iloc[Y_hat_insample.shape[0] - t_test-1], aux['ds'].iloc[aux.shape[0]-1], alpha=0.2, color='yellow', )

      # Se marcan las ventanas
      for cutoff in ventanas:
        ax[s].axvline(x=cutoff, color='black', ls=':')

      # Personalizacion de ejes
      ax[s].set(ylabel = 'Consumo de ' + w )

      # Personalizacion del titulo
      if s == 0:
        ax[s].set(title = 'Pronostico del consumo de medicamentos durante la primer ola de COVID-19.')

      plt.tight_layout()
      s = s + 1


# Iteracion sobre los medicamentos modelados
for w in lista_med:

  aux = datos_eval.copy()
  aux['y'] = aux[w].apply(lambda x: math.exp(x) if x > 0 else 0)

  # Caracteristicas del escenario retrospectivo
  re_entrenar = True
  t_val = 3
  t_test = 3
  t_step = 1
  t = t_step

  if w == 'FENTANIL':
    mod = 'mod_LSTM_Retro_1_MASE_2020_08_2020_39'
    aux_loss = 'MASE'
    l = 'LSTM'

  elif w == 'MIDAZOLAM':
    mod = 'mod_Autoformer_Retro_1_MASE_2020_08_2020_39'
    aux_loss = 'MASE'
    l = 'Autoformer'

  elif w == 'PARACETAMOL':
    mod = 'mod_TCN_Retro_1_MASE_2020_08_2020_39'
    aux_loss = 'MASE'
    l = 'TCN'

  # Inicializacion de las metricas de evaluacion
  aux_mape = 0
  aux_mase = 0
  aux_metr = 0
  aux_mean = 0

  # Se carga el modelo
  nf = NeuralForecast.load(path = '/' + mod)
  u = str(nf.models[0])

  # Se obtienen los pronosticos durante todo el entrenamiento, el periodo de prueba y el de validacion
  if re_entrenar:
    Y_hat_insample = pd.concat([nf.predict_insample(step_size = t_step)[-t_val-t_test+t_step-1:], nf.predict()])
    ventanas = pd.DatetimeIndex(Y_hat_insample[-t_val-t_test:]["cutoff"].unique()[:-1]).union(pd.DatetimeIndex([max(nf.predict_insample(step_size = t_step)["ds"])]))
    aux0 = aux['y'].values[:-t_step]

  else:
    Y_hat_insample = nf.predict_insample(step_size = t_step)[-t_val-t_test:]
    ventanas = pd.DatetimeIndex(Y_hat_insample["cutoff"].unique())
    aux0 = aux['y'].values[:-t_val-t_test]

  if aux_loss == 'QuantileLoss':
    Y_hat_insample[l] = Y_hat_insample[u + '_ql' + str(aux_ql)].apply(lambda x: math.exp(x) if x > 0 else 0)
  else:
    Y_hat_insample[l] = Y_hat_insample[u].apply(lambda x: math.exp(x) if x > 0 else 0)

  # Intervalo de validacion
  aux1 = aux['y'].values[-t_val:]
  aux2 = Y_hat_insample[l].values[-t_val:]

  # Se grafican los datos reales y los datos pronosticados
  ax[s].plot(aux['ds'], aux['y'])
  ax[s].plot(Y_hat_insample['ds'], Y_hat_insample[l], ls='--')

  # Se sombrean los intervalos de prueba y validacion
  ax[s].axvspan(ventanas[0], Y_hat_insample['ds'].iloc[Y_hat_insample.shape[0] - t_test-1], alpha=0.2, color='yellow', )
  ax[s].axvspan(Y_hat_insample['ds'].iloc[Y_hat_insample.shape[0] - t_test - 1], Y_hat_insample['ds'].iloc[Y_hat_insample.shape[0] - 1], alpha=0.2, color='red',)

  # Se marcan las ventanas
  for cutoff in ventanas:
    ax[s].axvline(x=cutoff, color='black', ls=':')

  # Personalizacion de ejes
  if s == 5:
    ax[s].set(xlabel = 'Semana epidemiológica')

  ax[s].set(ylabel = 'Consumo de ' + w )

  plt.tight_layout()

  s = s + 1

# Leyenda
leyenda_per = [Line2D([0], [0], color='tab:blue', label='Consumo real'),
                   Line2D([0], [0], color='tab:orange', label='Consumo pronosticado', ls='--'),
                   Patch(facecolor='yellow', edgecolor='b', label='Periodo de validación', alpha=0.2),
                   Patch(facecolor='tab:red', edgecolor='b', label='Periodo de prueba', alpha=0.2)]

f.legend(handles=leyenda_per, loc='upper left')
plt.show()

In [ ]:
# Ola a analizar
aux_ola = ola_retro_2

aux_ola_fin = datetime.datetime.strptime(aux_ola[8:12] + ' ' +  aux_ola[13:17] + ' 1', "%Y %W %w")
aux_ola_ini = datetime.datetime.strptime(aux_ola[0:4] + ' ' +  aux_ola[5:7] + ' 1', "%Y %W %w")

# Particion de datos: entrenamiento y prueba
aux = datos_series.loc[(aux_ola_ini <= datos_series['FECHA']) & (datos_series['FECHA'] <= aux_ola_fin)].copy()
aux_fecha = aux['FECHA'].copy().reset_index(drop=True)

# Todas las variables para el entrenamiento
all_var = [ruta_med] + lista_exog

# Todas las variables para la evaluacion
all_var2 = ["PARACETAMOL", "MIDAZOLAM", "FENTANIL"] + lista_exog

# Datos para la evaluacion de los modelos
datos_eval = pd.DataFrame(aux[all_var2]).reset_index(drop=True)
datos_eval['ds'] = aux_fecha
datos_eval['unique_id'] = '1'

# Tabla con resultados
tabla_res = pd.DataFrame(columns = ["MEDICAMENTO", "MODELOS", "RE_ENTRENO", "TIEMPO_PRONOSTICO", "METRICA_EVALUACION", "MASE", "ERROR_PROMEDIO"])

# Visualizacion al ajuste hecho a los datos de entrenamiento
f, ax = plt.subplots(6, figsize=(15, 15))
s = 0

#Iteracion sobre los medicamentos modelados
for w in lista_med:

  for l in ["AutoARIMA", "AutoETS"]:

    aux = datos_eval.copy()
    aux['y'] = aux[w].apply(lambda x: math.exp(x) if x > 0 else 0)

    # Caracteristicas del escenario retrospectivo
    re_entrenar = True
    t_val = 8
    t_test = 8
    t_step = 1
    t = t_step

    if l == "AutoARIMA":
      aux_loss = 'AIC'
      sf = StatsForecast(models=[AutoARIMA(seasonal=True, ic='aicc', stationary=False)], freq='W-Mon')

      # Se obtienen los pronosticos durante el periodo de prueba
      Y_hat_insample = sf.cross_validation(h = t, df = datos_eval[['ds', w, 'unique_id']][:-t_test-t_val], step_size= t_step, test_size= t_test, refit= re_entrenar, fitted = True, target_col=w  )
      for r in range(t_val+t_test+1):
        if r == 0:
          aux_for = sf.forecast(h = t, df = datos_eval[['ds', w, 'unique_id']][:-t_test-t_val+r-1], target_col=w)
        else:
          aux_for = pd.concat([aux_for, sf.forecast(h = t, df = datos_eval[['ds', w, 'unique_id']][:-t_test-t_val-1+r], target_col=w)], ignore_index=True)
      ventanas = aux_for['ds']
      aux_real = aux[['ds', w, 'unique_id']]
      aux_for[l] = aux_for[l].apply(lambda x: math.exp(x) if x > 0 else 0)
      aux_real = pd.merge(aux_real, aux_for, on='ds', how='left')
      aux0 = aux['y'].values[:-t_test]

      # Intervalo de prueba
      aux1 = aux['y'].values[-t_test:]
      aux2 = aux_real[l].values[-t_test:]

    # Modelos de NeuralForecast
    elif l == "AutoETS":
      aux_loss = 'AIC'
      sf = StatsForecast(models=[AutoETS(model="ANN", )], freq='W-Mon')

      # Se obtienen los pronosticos durante el periodo de prueba
      Y_hat_insample = sf.cross_validation(h = t, df = datos_eval[['ds', w, 'unique_id']][:-t_test-t_val], step_size= t_step, test_size= t_test, refit= re_entrenar, fitted = True, target_col=w  )
      for r in range(t_val+t_test+1):
        if r == 0:
          aux_for = sf.forecast(h = t, df = datos_eval[['ds', w, 'unique_id']][:-t_test-t_val+r-1], target_col=w)
        else:
          aux_for = pd.concat([aux_for, sf.forecast(h = t, df = datos_eval[['ds', w, 'unique_id']][:-t_test-t_val-1+r], target_col=w)], ignore_index=True)
      ventanas = aux_for['ds'] # La fecha del periodo en turno es el fin de la ventana del proximo
      aux_real = aux[['ds', w, 'unique_id']]
      aux_for[l] = aux_for[l].apply(lambda x: math.exp(x) if x > 0 else 0)
      aux_real = pd.merge(aux_real, aux_for, on='ds', how='left')
      aux0 = aux['y'].values[:-t_test]

      # Intervalo de prueba (porque no hay intervalo de validacion en la funcion implementada en la libreria)
      aux1 = aux['y'].values[-t_test:]
      aux2 = aux_real[l].values[-t_test:]


    # Se grafican los datos reales y los datos pronosticados
    if (l == 'AutoETS' and w == 'FENTANIL') or (l == 'AutoETS' and w == 'MIDAZOLAM') or (l == 'AutoETS' and w == 'PARACETAMOL'):

      ax[s].plot(aux['ds'], aux['y'])
      ax[s].plot(aux_real['ds'], aux_real[l], label=l, ls='--')

      # Se sombrean los intervalos de prueba y validacion
      ax[s].axvspan(Y_hat_insample['ds'].iloc[Y_hat_insample.shape[0] - t_test-1], aux['ds'].iloc[aux.shape[0]-1], alpha=0.2, color='yellow', label='Conjunto de validación')

      # Se marcan las ventanas
      for cutoff in ventanas:
        ax[s].axvline(x=cutoff, color='black', ls=':')

      # Personalizacion de ejes
      if s == 2:
        ax[s].set(xlabel = 'Semana epidemiológica')

      ax[s].set(ylabel = 'Consumo de ' + w )

      # Personalizacion del titulo
      if s == 0:
        ax[s].set(title = 'Pronostico del consumo de medicamentos durante la aplicación de la vacuna contra COVID-19 a mayores de 18 años.')

      plt.tight_layout()
      s = s + 1

s = 3

#Iteracion sobre los medicamentos modelados
for w in lista_med:

  aux = datos_eval.copy()
  aux['y'] = aux[w].apply(lambda x: math.exp(x) if x > 0 else 0)

  # Caracteristicas del escenario retrospectivo
  re_entrenar = True
  t_step = 1
  t_val = 8
  t_test = 8
  t = t_step

  if w == 'FENTANIL':
    mod = 'mod_LSTM_Retro_2_MASE_2020_12_2021_35'
    aux_loss = 'MASE'
    l = 'LSTM'

  elif w == 'MIDAZOLAM':
    mod = 'mod_Autoformer_Retro_2_MASE_2020_12_2021_35'
    aux_loss = 'MASE'
    l = 'Autoformer'

  elif w == 'PARACETAMOL':
    mod = 'mod_TCN_Retro_2_MASE_2020_12_2021_35'
    aux_loss = 'MASE'
    l = 'TCN'

  # Se carga el modelo
  nf = NeuralForecast.load(path = ruta_mod + '/' + w + '/' + mod)
  u = str(nf.models[0])

  # Se obtienen los pronosticos durante todo el entrenamiento, el periodo de prueba y el de validacion
  if re_entrenar:
    Y_hat_insample = pd.concat([nf.predict_insample(step_size = t_step)[-t_val-t_test+t_step-1:], nf.predict()])
    ventanas = pd.DatetimeIndex(Y_hat_insample[-t_val-t_test:]["cutoff"].unique()[:-1]).union(pd.DatetimeIndex([max(nf.predict_insample(step_size = t_step)["ds"])]))
    aux0 = aux['y'].values[:-t_step]

  else:
    Y_hat_insample = nf.predict_insample(step_size = t_step)[-t_val-t_test:]
    ventanas = pd.DatetimeIndex(Y_hat_insample["cutoff"].unique())
    aux0 = aux['y'].values[:-t_val-t_test]

  if aux_loss == 'QuantileLoss':
    Y_hat_insample[l] = Y_hat_insample[u + '_ql' + str(aux_ql)].apply(lambda x: math.exp(x) if x > 0 else 0)
  else:
    Y_hat_insample[l] = Y_hat_insample[u].apply(lambda x: math.exp(x) if x > 0 else 0)

  # Intervalo de validacion
  aux1 = aux['y'].values[-t_val:]
  aux2 = Y_hat_insample[l].values[-t_val:]

  # Se grafican los datos reales y los datos pronosticados
  ax[s].plot(aux['ds'], aux['y'])
  ax[s].plot(Y_hat_insample['ds'], Y_hat_insample[l], label=l, ls='--')

  # Se sombrean los intervalos de prueba y validacion
  ax[s].axvspan(ventanas[0], Y_hat_insample['ds'].iloc[Y_hat_insample.shape[0] - t_test-1], alpha=0.2, color='yellow', label='Conjunto de validación')
  ax[s].axvspan(Y_hat_insample['ds'].iloc[Y_hat_insample.shape[0] - t_test - 1], Y_hat_insample['ds'].iloc[Y_hat_insample.shape[0] - 1], alpha=0.2, color='red', label='Conjunto de prueba')

  # Se marcan las ventanas
  for cutoff in ventanas:
    ax[s].axvline(x=cutoff, color='black', ls=':')

  # Personalizacion de ejes
  if s == 5:
    ax[s].set(xlabel = 'Semana epidemiológica')

  ax[s].set(ylabel = 'Consumo de ' + w )

  # Personalizacion del titulo

  plt.tight_layout()

  s = s + 1

# Leyenda
leyenda_per = [Line2D([0], [0], color='tab:blue', label='Consumo real'),
                   Line2D([0], [0], color='tab:orange', label='Consumo pronosticado', ls='--'),
                   Patch(facecolor='yellow', edgecolor='b', label='Periodo de validación', alpha=0.2),
                   Patch(facecolor='tab:red', edgecolor='b', label='Periodo de prueba', alpha=0.2)]

f.legend(handles=leyenda_per, loc='upper left')
plt.show()



In [ ]:
# Ola a analizar
aux_ola = ola_retro_3

aux_ola_fin = datetime.datetime.strptime(aux_ola[8:12] + ' ' +  aux_ola[13:17] + ' 1', "%Y %W %w")
aux_ola_ini = datetime.datetime.strptime(aux_ola[0:4] + ' ' +  aux_ola[5:7] + ' 1', "%Y %W %w")

# Particion de datos: entrenamiento y prueba
aux = datos_series.loc[(aux_ola_ini <= datos_series['FECHA']) & (datos_series['FECHA'] <= aux_ola_fin)].copy()
aux_fecha = aux['FECHA'].copy().reset_index(drop=True)

# Todas las variables para el entrenamiento
all_var = [ruta_med] + lista_exog

# Todas las variables para la evaluacion
all_var2 = ["PARACETAMOL", "MIDAZOLAM", "FENTANIL"] + lista_exog

# Datos para la evaluacion de los modelos
datos_eval = pd.DataFrame(aux[all_var2]).reset_index(drop=True)
datos_eval['ds'] = aux_fecha
datos_eval['unique_id'] = '1'

# Tabla con resultados
tabla_res = pd.DataFrame(columns = ["MEDICAMENTO", "MODELOS", "RE_ENTRENO", "TIEMPO_PRONOSTICO", "METRICA_EVALUACION", "MASE", "ERROR_PROMEDIO"])

# Visualizacion al ajuste hecho a los datos de entrenamiento
f, ax = plt.subplots(6, figsize=(15, 15))
s = 0

#Iteracion sobre los medicamentos modelados
for w in lista_med:

  for l in ["AutoARIMA", "AutoETS"]:

    aux = datos_eval.copy()
    aux['y'] = aux[w].apply(lambda x: math.exp(x) if x > 0 else 0)

    # Caracteristicas del escenario retrospectivo
    re_entrenar = True
    t_val = 8
    t_test = 8
    t_step = 1
    t = t_step

    if l == "AutoARIMA":
      aux_loss = 'AIC'
      sf = StatsForecast(models=[AutoARIMA(seasonal=True, ic='aicc', stationary=False)], freq='W-Mon')

      # Se obtienen los pronosticos durante el periodo de prueba
      Y_hat_insample = sf.cross_validation(h = t, df = datos_eval[['ds', w, 'unique_id']][:-t_test-t_val], step_size= t_step, test_size= t_test, refit= re_entrenar, fitted = True, target_col=w  )
      for r in range(t_val+t_test+1):
        if r == 0:
          aux_for = sf.forecast(h = t, df = datos_eval[['ds', w, 'unique_id']][:-t_test-t_val+r-1], target_col=w)
        else:
          aux_for = pd.concat([aux_for, sf.forecast(h = t, df = datos_eval[['ds', w, 'unique_id']][:-t_test-t_val-1+r], target_col=w)], ignore_index=True)
      ventanas = aux_for['ds']
      aux_real = aux[['ds', w, 'unique_id']]
      aux_for[l] = aux_for[l].apply(lambda x: math.exp(x) if x > 0 else 0)
      aux_real = pd.merge(aux_real, aux_for, on='ds', how='left')
      aux0 = aux['y'].values[:-t_test]

      # Intervalo de prueba
      aux1 = aux['y'].values[-t_test:]
      aux2 = aux_real[l].values[-t_test:]

    # Modelos de NeuralForecast
    elif l == "AutoETS":
      aux_loss = 'AIC'
      sf = StatsForecast(models=[AutoETS(model="ANN", )], freq='W-Mon')

      # Se obtienen los pronosticos durante el periodo de prueba
      Y_hat_insample = sf.cross_validation(h = t, df = datos_eval[['ds', w, 'unique_id']][:-t_test-t_val], step_size= t_step, test_size= t_test, refit= re_entrenar, fitted = True, target_col=w  )
      for r in range(t_val+t_test+1):
        if r == 0:
          aux_for = sf.forecast(h = t, df = datos_eval[['ds', w, 'unique_id']][:-t_test-t_val+r-1], target_col=w)
        else:
          aux_for = pd.concat([aux_for, sf.forecast(h = t, df = datos_eval[['ds', w, 'unique_id']][:-t_test-t_val-1+r], target_col=w)], ignore_index=True)
      ventanas = aux_for['ds']
      aux_real = aux[['ds', w, 'unique_id']]
      aux_for[l] = aux_for[l].apply(lambda x: math.exp(x) if x > 0 else 0)
      aux_real = pd.merge(aux_real, aux_for, on='ds', how='left')
      aux0 = aux['y'].values[:-t_test]

      # Intervalo de prueba
      aux1 = aux['y'].values[-t_test:]
      aux2 = aux_real[l].values[-t_test:]


    # Se grafican los datos reales y los datos pronosticados
    if (l == 'AutoETS' and w == 'FENTANIL') or (l == 'AutoETS' and w == 'MIDAZOLAM') or (l == 'AutoETS' and w == 'PARACETAMOL'):

      ax[s].plot(aux['ds'], aux['y'])
      ax[s].plot(aux_real['ds'], aux_real[l], label=l, ls='--')

      # Se sombrean los intervalos de prueba y validacion
      ax[s].axvspan(Y_hat_insample['ds'].iloc[Y_hat_insample.shape[0] - t_test-1], aux['ds'].iloc[aux.shape[0]-1], alpha=0.2, color='yellow', label='Conjunto de validación')

      # Se marcan las ventanas
      for cutoff in ventanas:
        ax[s].axvline(x=cutoff, color='black', ls=':')

      # Personalizacion de ejes
      if s == 2:
        ax[s].set(xlabel = 'Semana epidemiológica')

      ax[s].set(ylabel = 'Consumo de ' + w )

      # Personalizacion del titulo
      if s == 0:
        ax[s].set(title = 'Pronostico del consumo de medicamentos durante cepa Omicron de Covid-19.')

      plt.tight_layout()
      s = s + 1

s = 3

#Iteracion sobre los medicamentos modelados
for w in lista_med:

  aux = datos_eval.copy()
  aux['y'] = aux[w].apply(lambda x: math.exp(x) if x > 0 else 0)

  # Caracteristicas del escenario retrospectivo
  re_entrenar = True
  t_step = 1
  t_val = 8
  t_test = 8
  t = t_step

  if w == 'FENTANIL':
    mod = 'mod_LSTM_Retro_3_MASE_2020_12_2022_09'
    aux_loss = 'MASE'
    l = 'LSTM'

  elif w == 'MIDAZOLAM':
    mod = 'mod_Autoformer_Retro_3_MASE_2020_12_2022_09'
    aux_loss = 'MASE'
    l = 'Autoformer'

  elif w == 'PARACETAMOL':
    mod = 'mod_TCN_Retro_3_MASE_2020_12_2022_09'
    aux_loss = 'MASE'
    l = 'TCN'

  # Se carga el modelo
  nf = NeuralForecast.load(path = ruta_mod + '/' + w + '/' + mod)
  u = str(nf.models[0])

  # Se obtienen los pronosticos durante todo el entrenamiento, el periodo de prueba y el de validacion
  if re_entrenar:
    Y_hat_insample = pd.concat([nf.predict_insample(step_size = t_step)[-t_val-t_test+t_step-1:], nf.predict()])
    ventanas = pd.DatetimeIndex(Y_hat_insample[-t_val-t_test:]["cutoff"].unique()[:-1]).union(pd.DatetimeIndex([max(nf.predict_insample(step_size = t_step)["ds"])]))
    aux0 = aux['y'].values[:-t_step]

  else:
    Y_hat_insample = nf.predict_insample(step_size = t_step)[-t_val-t_test:]
    ventanas = pd.DatetimeIndex(Y_hat_insample["cutoff"].unique())
    aux0 = aux['y'].values[:-t_val-t_test]

  if aux_loss == 'QuantileLoss':
    Y_hat_insample[l] = Y_hat_insample[u + '_ql' + str(aux_ql)].apply(lambda x: math.exp(x) if x > 0 else 0)
  else:
    Y_hat_insample[l] = Y_hat_insample[u].apply(lambda x: math.exp(x) if x > 0 else 0)

  # Intervalo de validacion
  aux1 = aux['y'].values[-t_val:]
  aux2 = Y_hat_insample[l].values[-t_val:]

  # Se grafican los datos reales y los datos pronosticados
  ax[s].plot(aux['ds'], aux['y'])
  ax[s].plot(Y_hat_insample['ds'], Y_hat_insample[l], label=l, ls='--')

  # Se sombrean los intervalos de prueba y validacion
  ax[s].axvspan(ventanas[0], Y_hat_insample['ds'].iloc[Y_hat_insample.shape[0] - t_test-1], alpha=0.2, color='yellow', label='Conjunto de validación')
  ax[s].axvspan(Y_hat_insample['ds'].iloc[Y_hat_insample.shape[0] - t_test - 1], Y_hat_insample['ds'].iloc[Y_hat_insample.shape[0] - 1], alpha=0.2, color='red', label='Conjunto de prueba')

  # Se marcan las ventanas
  for cutoff in ventanas:
    ax[s].axvline(x=cutoff, color='black', ls=':')

  # Personalizacion de ejes
  if s == 5:
    ax[s].set(xlabel = 'Semana epidemiológica')

  ax[s].set(ylabel = 'Consumo de ' + w )

  plt.tight_layout()

  s = s + 1

# Leyenda
leyenda_per = [Line2D([0], [0], color='tab:blue', label='Consumo real'),
                   Line2D([0], [0], color='tab:orange', label='Consumo pronosticado', ls='--'),
                   Patch(facecolor='yellow', edgecolor='b', label='Periodo de validación', alpha=0.2),
                   Patch(facecolor='tab:red', edgecolor='b', label='Periodo de prueba', alpha=0.2)]

f.legend(handles=leyenda_per, loc='upper left')
plt.show()
